In [ ]:
%load_ext autoreload
%autoreload 2
# import libraries
import numpy as np
import pandas as pd
import os
import time
import yaml
from pathlib import Path
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from multiprocessing import Pool, cpu_count
from functools import reduce

import pyhealth
import pyhealth.datasets.mimic4 as mimic4
print(pyhealth.__file__)

import polars as pl


In [ ]:
def load_config(config_path='../configs/preprocessing.yml'):
    # Guard check - Config Exists
    config_path = Path(config_path)
    if not config_path.exists():
        raise FileNotFoundError(f"Configuration file not found at: {config_path.resolve()}")

    try:
        with open(config_path, 'r') as f:
            full_cfg = yaml.safe_load(f)

        cfg = full_cfg['MIMIC']

        root_dir = Path(cfg['root_dir']).resolve()
        mimic_dir = root_dir / cfg['mimic_subdir']
        cxr_dir = root_dir / cfg['cxr_subdir']
        res_dir = root_dir / cfg['output_subdir']

        root_files = cfg['root_files']
        patients_csv = root_dir / root_files['patients']
        admissions_csv = root_dir / root_files['admissions']
        chexpert_csv = root_dir / root_files['chexpert']
        metadata_csv = root_dir / root_files['metadata']

        return {
            'root_dir': mimic_dir,
            'cxr_dir': cxr_dir,
            'mimic_dir': mimic_dir,
            'out_dir': res_dir,
            'patients_csv': patients_csv,
            'admissions_csv': admissions_csv,
            'chexpert_csv': chexpert_csv,
            'metadata_csv': metadata_csv
        }
    except Exception as e:
        raise Exception(f"Error loading preprocessing configuration {e} for config {config_path.resolve()}")
    
config = load_config()


In [ ]:
def load_and_merge_mimic_data_pyhealth():
    ds = mimic4.MIMIC4Dataset(
        cxr_root=config['cxr_dir'],
        cxr_tables=["chexpert", "metadata"],
        cxr_config_path=Path("../configs/pyhealth/mimic4_cxr.yaml").resolve(),
        ehr_root=config['mimic_dir'],
        ehr_tables=["patients", "admissions"],
        ehr_config_path=Path("../configs/pyhealth/mimic4_ehr.yaml").resolve(),
    )
    md = ds.sub_datasets["cxr"].load_table("metadata")
    pt = ds.sub_datasets["ehr"].load_table("patients")
    ad = ds.sub_datasets["ehr"].load_table("admissions")
    lb = ds.sub_datasets["cxr"].load_table("chexpert")

    base = (
        md.filter(pl.col("metadata/viewposition").is_in(["PA", "AP"]))
          .select([
              pl.col("metadata/dicom_id").alias("dicom_id"),
              pl.col("patient_id").alias("subject_id"),
              pl.col("metadata/study_id").alias("study_id"),
              pl.col("metadata/viewposition").alias("ViewPosition"),
              pl.col("metadata/image_path").alias("path")
          ])
    )

    ehr_joins = {
        "gender": pt.select(
            pl.col("patient_id").alias("subject_id"),
            pl.col("patients/gender").alias("gender")
        ),
        "insurance_ethnicity_status": (
            ad.select(
                pl.col("patient_id").alias("subject_id"),
                pl.col("admissions/insurance").alias("insurance"),
                pl.col("admissions/race").alias("ethnicity"),
                pl.col("admissions/marital_status").alias("marital_status")
            )
            .unique(subset=["subject_id"], keep="last")
        )
    }
    base = reduce(lambda df, tbl: df.join(
        tbl, on="subject_id", how="left"), ehr_joins.values(), base)

    label_cols = [
        "atelectasis", "cardiomegaly", "consolidation", "edema",
        "enlarged cardiomediastinum", "fracture", "lung lesion",
        "lung opacity", "no finding", "pleural effusion",
        "pleural other", "pneumonia", "pneumothorax", "support devices"
    ]
    rename_map = {f"chexpert/{c}": c.title() for c in label_cols}
    joined = base.join(
        lb.rename(rename_map),
        left_on="dicom_id",
        right_on="chexpert/dicom_id",
        how="left"
    )

    df = (
        joined
        .filter(
            pl.col("gender").is_not_null() &
            pl.col("insurance").is_not_null() &
            pl.col("ethnicity").is_not_null()
        )
        .select(
            ["dicom_id", "subject_id", "study_id", "ViewPosition", "gender",
             "insurance", "ethnicity", "marital_status"] +
            [c.title() for c in label_cols] +
            ["path"]
        )
        .collect()
    )
    return df.to_pandas()

In [ ]:
def process_image(img, transResize):
    # Center crop the image to square dimensions
    width, height = img.size
    r_min = max(0, (height - width) / 2)
    r_max = min(height, (height + width) / 2)
    c_min = max(0, (width - height) / 2)
    c_max = min(width, (width + height) / 2)
    img = img.crop((c_min, r_min, c_max, r_max))

    # Resize the image to the target size (transResize x transResize)
    img = img.resize((transResize, transResize))

    # Equalize histogram for contrast enhancement and convert to grayscale
    img = ImageOps.equalize(img)
    img = img.convert('L')

    return np.array(img)

# Loads and preprocesses a single image given its index and path.
# Returns a tuple of (index, processed_image_array), or None if any step fails.
def worker_process(index, filename, transResize, data_dir):
    try:
        img_path = os.path.join(data_dir, filename)
        img = Image.open(img_path).convert('RGB')
        img_array = process_image(img, transResize)
        return index, img_array
    except Exception as e:
        return None


def processAndSave(df, transResize=128, pool_size=28, chunk_size=1000):
    # Load preprocessing config
    config = load_config()
    
    # Prepare a memory-mapped .npy file to store resized image data efficiently;
    # creates the file if it doesn't exist, otherwise opens it for read/write access
    npy_output_path = os.path.join(config['out_dir'], f'files_{transResize}.npy')
    print(npy_output_path)
    os.makedirs(os.path.dirname(npy_output_path), exist_ok=True)
    if not os.path.exists(npy_output_path):
        img_mat = np.memmap(npy_output_path, dtype='uint8', mode='w+', shape=(len(df), transResize, transResize))
    else:
        img_mat = np.memmap(npy_output_path, dtype='uint8', mode='r+', shape=(len(df), transResize, transResize))

    valid_indices = []
    processed_count = 0

    # Prepare arguments for multiprocessing
    args = [(i, fname, transResize, config['mimic_dir']) for i, fname in enumerate(df['path'])]

    with Pool(pool_size) as pool:
        pbar = tqdm(range(0, len(args), chunk_size), desc="Processing", dynamic_ncols=True)
        for i in pbar:
            chunk_start = time.perf_counter()

            batch = args[i:i + chunk_size]
            results = pool.starmap(worker_process, batch)

            for res in results:
                if res is None:
                    continue
                idx, img_array = res
                img_mat[len(valid_indices)] = img_array
                valid_indices.append(idx)
                processed_count += 1

            chunk_elapsed = time.perf_counter() - chunk_start
            item_time = chunk_elapsed / len(batch) if len(batch) > 0 else 0

            pbar.set_postfix({
                "Batch Time": f"{chunk_elapsed:.2f}s",
                "Item Time": f"{item_time:.4f}s",
                "Total": processed_count
            })

    # Finalize: flush memmap and save filtered metadata
    img_mat.flush()
    df2 = df.iloc[valid_indices].reset_index(drop=True)
    df2.to_csv(os.path.join(config['out_dir'], 'meta_data.csv'), index=False)




In [ ]:
start = time.time()
df = load_and_merge_mimic_data_pyhealth()
processAndSave(df,pool_size=28)
print(f"✅ Completed in {round((time.time() - start) / 60, 2)} minutes")